In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. Introduction: Problem and Data Description

## 1.1. Problem Statement:

The primary objective of the Kaggle competition is to develop a Generative Adversarial Network (GAN) that can transform real-world photographs into digital images resembling the distinctive artistic style of Claude Monet's paintings.

## 1.2. Generative Deep Learning Models (GANs):

Generative Adversarial Networks (GANs) are a class of deep learning models composed of two competing neural networks:
* **Generator Network:** This network's role is to create synthetic data (in this case, Monet-style paintings from photos) that aims to be indistinguishable from real data. It learns to map random noise or an input image to a desired output distribution.
* **Discriminator Network:** This network acts as a "critic," attempting to distinguish between real data (actual Monet paintings) and fake data (images generated by the generator).

The training process is adversarial: the generator continuously tries to "fool" the discriminator into believing its generated images are real, while the discriminator strives to improve its ability to identify fake images. This competitive dynamic drives both networks to improve, ultimately leading the generator to produce highly realistic outputs.

## 1.3. Dataset Description:

The competition provides two main datasets:

* **Monet paintings:** This dataset consists of **300** original paintings by Claude Monet. These are available in both JPEG (*monet_jpg*) and TFRecord (*monet_tfrec*) formats.
* **Photos:** This dataset contains **7028** real-world photographs. These are also provided in both JPEG (*photo_jpg*) and TFRecord (*photo_tfrec*) formats.

Both datasets contain images with dimensions of 256x256 pixels. A crucial characteristic of these datasets for this competition is their unpaired nature. This means there is no direct, one-to-one correspondence or ground truth mapping between a specific photo and a Monet painting of the same scene. This unpaired characteristic is why architectures like CycleGAN are often favored for this challenge, as they are designed to handle image-to-image translation without requiring such paired training data.

# 2. Exploratory Data Analysis (EDA): Inspect, Visualize, and Clean

## 2.1. Image Inspection and Verification:

* **Loading and Displaying Samples:**

    To get an immediate visual understanding, we would typically write Python code using libraries like **matplotlib** and **tensorflow.data** (or **PIL**/**opencv**) to load and display a small subset (e.g., 5-10 images) from both the "Monet paintings" and "Photos" datasets. This step visually confirms the type of images we are working with and their distinct styles.

* **Dimension and Channel Check:**

    Programmatically, we would inspect the shape of a few loaded images. We expect images to have a consistent shape, typically (**height, width, channels**). For this competition, images are 256x256 pixels, and as color images, they should have 3 color channels (RGB). Verifying these dimensions ensures uniformity and compatibility with the input requirements of our neural networks.

## 2.2. Data Characteristics:

* **Visual Observations:**

    * **Monet Paintings:** Upon visual inspection, Monet's paintings typically exhibit a distinct Impressionistic style. We would observe:

        * **Color Palettes:** Often dominated by soft, blended hues, with a focus on capturing the effect of light. Colors might appear diffused rather than sharp.

        * **Common Subjects:** Frequent subjects include landscapes, water lilies (his famous series), haystacks, cathedrals, and scenes from daily life, often depicting outdoor settings.

        * **Overall Artistic Style:** Characterized by visible brushstrokes, open composition, emphasis on light in its changing qualities, and ordinary subject matter. The focus is often on the perception of light and color rather than clear, sharp outlines.

    * **Photos:** The photographic dataset, in contrast, would display:

        * **Variety of Scenes:** A wide array of real-world scenes, including landscapes, portraits, urban environments, and objects, reflecting diverse content.

        * **Lighting Conditions:** Varied lighting, from bright daylight to dusk, artificial lighting, and shadows, showcasing typical photographic diversity.

        * **Composition:** Traditional photographic compositions, with clear subjects, backgrounds, and distinct forms, unlike the impressionistic blur of Monet's work.

* **Dataset Sizes:**

    As confirmed in Section 1.3, the dataset sizes are:

    * **Monet paintings:** 300 images.

    * **Photos:** 7028 images.

## 2.3. Data Cleaning/Preprocessing:

* **Necessity for GANs:**

    Preprocessing is particularly vital for GANs because they are highly sensitive to the distribution and range of input data. Consistent and normalized input helps stabilize the training process, prevents issues like vanishing/exploding gradients, and allows the networks to learn meaningful features more effectively. Many GAN architectures, especially those using tanh activation in the generator's output layer, expect input pixel values in a specific range (e.g., [-1, 1]).

* **Specific Steps:**

    * **Resizing:** While the Kaggle competition explicitly states images are already 256x256, if they weren't uniform, this would be a crucial step. All images must be resized to a consistent dimension (e.g., 256x256 pixels) to match the expected input shape of the neural networks. This can involve cropping or padding if aspect ratios need to be preserved.

    * **Normalization:** This is a critical step for GANs. Pixel values, typically in the range [0, 255], need to be scaled to a specific range, most commonly [-1, 1]. This can be achieved using the formula:

        > Normalized_Pixel = (Pixel/127.5)−1

        This normalization aligns the input data with the output range of the tanh activation function, which is often used in the final layer of GAN generators.

    * **Augmentation (Optional but Recommended):** Data augmentation techniques are highly beneficial for GAN training to increase the diversity of the training data and prevent the generator from simply memorizing the training images (a problem known as "mode collapse"). Common augmentations include:

        * **Random Cropping:** Extracting random patches from the images.

        * **Horizontal Flipping:** Mirroring images horizontally.

        * **Random Jittering:** Slightly adjusting brightness, contrast, and saturation.

        * **Justification:** These techniques help the GAN learn more robust features and improve the generalization capability of the generator, making the generated images more diverse and less prone to overfitting the limited training data, especially for the Monet dataset with only 300 images.

In [2]:
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np
import os

# Define the paths to the datasets
MONET_PHOTO_DATASET_PATH = '/kaggle/input/monet-hand-painted-photos'
MONET_TRAIN_PATH = os.path.join(MONET_PHOTO_DATASET_PATH, 'monet_tfrec')
PHOTO_TRAIN_PATH = os.path.join(MONET_PHOTO_DATASET_PATH, 'photo_tfrec')

# Helper function to decode images from TFRecord files
def decode_image(image):
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.cast(image, tf.float32)
    # Ensure consistent shape after decoding
    image = tf.reshape(image, [256, 256, 3])
    # Normalize images to [-1, 1] as commonly used in GANs
    image = (image / 127.5) - 1
    return image

# Helper function to read TFRecord files
def read_tfrecord(example):
    tfrecord_format = {
        "image_name": tf.io.FixedLenFeature([], tf.string),
        "image": tf.io.FixedLenFeature([], tf.string),
        "target": tf.io.FixedLenFeature([], tf.string)
    }
    example = tf.io.parse_single_example(example, tfrecord_format)
    image = decode_image(example['image'])
    return image

# Function to load dataset from TFRecord files
def load_dataset(path, count=None, repeat=False, shuffle=False, batch_size=1):
    AUTO = tf.data.AUTOTUNE
    filenames = tf.io.gfile.glob(path + '/*.tfrec')
    if not filenames:
        print(f"Error: No TFRecord files found at {path}")
        return None # Return None if no files are found

    dataset = tf.data.TFRecordDataset(filenames)
    dataset = dataset.map(read_tfrecord, num_parallel_calls=AUTO)
    if shuffle:
        dataset = dataset.shuffle(2048) # Buffer size for shuffling
    if repeat:
        dataset = dataset.repeat()
    if count:
        dataset = dataset.take(count) # Take a specific number of samples
    dataset = dataset.batch(batch_size) # Batch for consistent output shape
    return dataset


### 2.1. Image Inspection and Verification:

print("--- 2.1. Image Inspection and Verification ---")

# Load a small sample of Monet paintings
# Use `unbatch()` and `take()` to get individual images for display
monet_dataset_for_display = load_dataset(MONET_TRAIN_PATH, batch_size=1)
monet_sample_images = []
if monet_dataset_for_display:
    for img_batch in monet_dataset_for_display.take(5): # Take 5 batches (each batch is 1 image)
        monet_sample_images.append(img_batch[0].numpy()) # Extract the image from the batch
else:
    print("Could not load Monet dataset for display.")


# Load a small sample of Photos
photo_dataset_for_display = load_dataset(PHOTO_TRAIN_PATH, batch_size=1)
photo_sample_images = []
if photo_dataset_for_display:
    for img_batch in photo_dataset_for_display.take(5): # Take 5 batches (each batch is 1 image)
        photo_sample_images.append(img_batch[0].numpy()) # Extract the image from the batch
else:
    print("Could not load Photo dataset for display.")

print("\nLoading and Displaying Sample Images:")

if monet_sample_images:
    plt.figure(figsize=(12, 6))
    plt.suptitle("Sample Monet Paintings", fontsize=16)
    for i, img in enumerate(monet_sample_images):
        plt.subplot(1, 5, i + 1)
        plt.imshow((img * 0.5 + 0.5)) # Denormalize for display [0, 1]
        plt.axis('off')
    plt.show()
else:
    print("No Monet images to display.")

if photo_sample_images:
    plt.figure(figsize=(12, 6))
    plt.suptitle("Sample Photos", fontsize=16)
    for i, img in enumerate(photo_sample_images):
        plt.subplot(1, 5, i + 1)
        plt.imshow((img * 0.5 + 0.5)) # Denormalize for display [0, 1]
        plt.axis('off')
    plt.show()
else:
    print("No Photo images to display.")

# Dimension and Channel Check
print("\nDimension and Channel Check:")
if monet_sample_images:
    first_monet_image = monet_sample_images[0]
    print(f"Monet Image Shape: {first_monet_image.shape}")
    print(f"Monet Image Data Type: {first_monet_image.dtype}")
    print(f"Monet Image Pixel Value Range: [{first_monet_image.min()}, {first_monet_image.max()}]")
else:
    print("No Monet images loaded for inspection.")

if photo_sample_images:
    first_photo_image = photo_sample_images[0]
    print(f"Photo Image Shape: {first_photo_image.shape}")
    print(f"Photo Image Data Type: {first_photo_image.dtype}")
    print(f"Photo Image Pixel Value Range: [{first_photo_image.min()}, {first_photo_image.max()}]")
else:
    print("No Photo images loaded for inspection.")

print("\n--- 2.2. Data Characteristics ---")
# Dataset sizes are already known from Section 1.3
print(f"Monet paintings dataset size: 300 images")
print(f"Photos dataset size: 7028 images")


### 2.3. Data Cleaning/Preprocessing:

print("\n--- 2.3. Data Cleaning/Preprocessing ---")

# Define preprocessing steps as a TensorFlow function for efficiency
# This function will be applied to each image in the dataset
IMG_WIDTH = 256
IMG_HEIGHT = 256

def preprocess_image_train(image):
    # Augmentation: Random horizontal flip
    image = tf.image.random_flip_left_right(image)

    # Augmentation: Random jittering (brightness, contrast, hue, saturation)
    # Applying jitter *before* the final -1 to 1 normalization is generally better
    # as it operates on a more intuitive intensity scale [0, 255] (before the final -1, 1 step).
    # However, our decode_image already normalizes to [-1, 1].
    # So, we need to convert back to [0, 1] or [0, 255] for jittering, then re-normalize.
    # Let's adjust to perform jittering on [0, 1] and then re-normalize to [-1, 1].

    # Denormalize to [0, 1] for jittering
    image = (image * 0.5) + 0.5

    # Apply jittering
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    image = tf.image.random_saturation(image, lower=0.8, upper=1.2)
    image = tf.image.random_hue(image, max_delta=0.08)

    # Clip values to ensure they stay within [0, 1] after jittering
    image = tf.clip_by_value(image, 0, 1)

    # Renormalize back to [-1, 1]
    image = (image * 2) - 1

    return image

print("Preprocessing Function Defined: `preprocess_image_train`")
print("Steps included:")
print("- Random Horizontal Flipping")
print("- Random Brightness Adjustment (max_delta=0.2)")
print("- Random Contrast Adjustment (lower=0.8, upper=1.2)")
print("- Random Saturation Adjustment (lower=0.8, upper=1.2)")
print("- Random Hue Adjustment (max_delta=0.08)")
print("- Pixel values are denormalized to [0,1] for jittering and then re-normalized to [-1, 1].")

2025-07-28 16:21:59.088726: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753719719.111367     120 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753719719.118443     120 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


--- 2.1. Image Inspection and Verification ---
Error: No TFRecord files found at /kaggle/input/monet-hand-painted-photos/monet_tfrec
Could not load Monet dataset for display.
Error: No TFRecord files found at /kaggle/input/monet-hand-painted-photos/photo_tfrec
Could not load Photo dataset for display.

Loading and Displaying Sample Images:
No Monet images to display.
No Photo images to display.

Dimension and Channel Check:
No Monet images loaded for inspection.
No Photo images loaded for inspection.

--- 2.2. Data Characteristics ---
Monet paintings dataset size: 300 images
Photos dataset size: 7028 images

--- 2.3. Data Cleaning/Preprocessing ---
Preprocessing Function Defined: `preprocess_image_train`
Steps included:
- Random Horizontal Flipping
- Random Brightness Adjustment (max_delta=0.2)
- Random Contrast Adjustment (lower=0.8, upper=1.2)
- Random Saturation Adjustment (lower=0.8, upper=1.2)
- Random Hue Adjustment (max_delta=0.08)
- Pixel values are denormalized to [0,1] for ji

## 2.4. Analysis Plan/Hypotheses:

* Initial Thoughts on Architecture:

    Given the unpaired nature of the "Monet paintings" and "Photos" datasets (i.e., no direct photo-to-Monet correspondences), a CycleGAN architecture is hypothesized to be the most suitable choice. CycleGAN is specifically designed for unpaired image-to-image translation by introducing a "cycle consistency" loss, which ensures that translating an image from domain A to B and then back to A results in an image similar to the original. This circumvents the need for paired data.

* Expected Challenges:

    Training GANs, especially CycleGANs, often presents several challenges:

    * Mode Collapse: The generator might learn to produce only a limited variety of outputs that consistently fool the discriminator, ignoring other modes in the true data distribution, leading to a lack of diversity in generated images.

    * Training Instability: The adversarial training process can be inherently unstable, making it difficult to find a balanced equilibrium between the generator and discriminator. This can lead to oscillating losses or one network overpowering the other.

    * Long Training Times: GANs, especially for high-resolution image generation, typically require significant computational resources and long training durations (many epochs) to achieve high-quality results.

    * Hyperparameter Sensitivity: GAN performance is often highly sensitive to hyperparameter choices (e.g., learning rates, network depths, loss function weights).

print("\n--- 2.4. Analysis Plan/Hypotheses ---")
print("Initial Thoughts on Architecture: CycleGAN is the primary candidate due to the unpaired nature of the datasets.")
print("Expected Challenges: Mode collapse, training instability, long training times, and hyperparameter sensitivity.")

# Example of how you would apply preprocessing to a dataset (not executed here for brevity of EDA output)
# train_monet_dataset_preprocessed = load_dataset(MONET_TRAIN_PATH, repeat=True, shuffle=True).map(preprocess_image_train)
# train_photo_dataset_preprocessed = load_dataset(PHOTO_TRAIN_PATH, repeat=True, shuffle=True).map(preprocess_image_train)

# 3. Model Architecture

# 3.1. Xxx

3. Model Architecture (25 pts)

GAN Type:

Specify the type of GAN architecture you've chosen (e.g., CycleGAN, Pix2Pix, DCGAN, or a custom architecture). Justify your choice, particularly if it's suitable for unpaired image-to-image translation (like CycleGAN).

Generator Network:

Describe the architecture of your generator (e.g., U-Net based, sequence of downsampling and upsampling convolutional layers, activation functions, batch normalization).

Explain its role in transforming input photos into Monet-style images.

Discriminator Network:

Describe the architecture of your discriminator (e.g., PatchGAN, sequence of convolutional layers).

Explain its role in distinguishing between real Monet paintings and generated Monet-style images.

Loss Functions:

Explain the loss functions used for both the generator and discriminator (e.g., adversarial loss, cycle consistency loss for CycleGAN, identity loss).

Discuss how these losses contribute to the GAN's training objective.

Optimizer:

Mention the optimizer chosen (e.g., Adam) and its parameters (learning rate, betas).

References:

Cite any research papers, Kaggle notebooks, or tutorials that informed your architectural decisions or GAN implementation.

# 4. Results and Analysis

# 4.1. Yyy

4. Results and Analysis (35 pts)

Training Process:

Describe your training setup (e.g., number of epochs, batch size, hardware used).

Visualize training progress (e.g., plots of generator and discriminator losses over epochs).

Hyperparameter Tuning & Architecture Comparison:

Discuss different hyperparameters you experimented with (e.g., learning rates, network depths, different types of normalization layers).

If you tried different GAN variants or significant architectural changes, compare their performance and discuss why some worked better than others.

Performance Improvement Techniques:

Describe any techniques used to improve training stability or image quality (e.g., spectral normalization, instance normalization, gradient penalization, different initialization strategies).

Discuss what helped and what didn't.

Generated Image Samples:

Display a selection of generated Monet-style images.

Compare them visually with both the original photos and real Monet paintings.

Quantitative Evaluation (MiFID Score):

Explain the MiFID (Memorization-informed Fréchet Inception Distance) metric. Briefly explain how it's an improvement over standard FID by penalizing memorization.

Present your achieved MiFID score from the Kaggle leaderboard.

Analyze your score in the context of the competition and discuss what it indicates about the quality and diversity of your generated images.

Troubleshooting:

Document any challenges encountered during training (e.g., mode collapse, training instability) and how you attempted to address them.

# 5. Conclusion

5. Conclusion (15 pts)

Interpretation of Results: Discuss your model's strengths and weaknesses based on the generated images and your MiFID score.

Key Learnings: Summarize what you learned about GANs, image generation, and the challenges of training such models.

Improvements: Detail specific architectural or training improvements that positively impacted your results, and mention those that did not.

Future Work: Suggest next steps for improving the model, such as exploring different GAN architectures, using larger datasets, or implementing more advanced training techniques.